# 003 Create Deep Agent

这是 Deep Agents 学习线的第三份 Notebook。

上一课产物：

- 一个可用的 `internet_search` 工具
- 理解了 Deep Agents 对工具的要求

本课产物：

- 一个配置好 model 和 tools 的 Deep Agent
- 理解 `create_deep_agent` 的参数

配套官方文档：

- [Quickstart - Create Agent](https://docs.langchain.com/oss/python/deepagents/quickstart)

学习目标：

1. 理解 `create_deep_agent` 的核心参数。
2. 创建 LangChain 模型实例并配置 tool calling。
3. 使用 `create_deep_agent` 创建 agent。
4. 理解 agent 的 system prompt 设计。

## 0. 加载项目配置

统一从 `.env` 读取配置。

In [1]:
import os
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / 'requirements.txt').exists() and (path / 'notebooks').exists():
            return path
    return current

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env', override=False)

MODEL_CONFIG = {
    'api_key': os.getenv('OPENAI_API_KEY', 'EMPTY'),
    'model': os.getenv('OPENAI_MODEL', 'qwq'),
    'base_url': os.getenv('OPENAI_BASE_URL', 'http://192.168.102.19:8082/v1'),
}

safe_config = dict(MODEL_CONFIG)
safe_config['api_key'] = '***'
pprint(safe_config)

{'api_key': '***', 'base_url': 'http://192.168.102.19:8082/v1', 'model': 'qwq'}


## 1. 创建 LangChain 模型实例

Deep Agents 需要一个支持 tool calling 的模型。

这里用 `ChatOpenAI` 连接 OpenAI-compatible 网关。

In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model=MODEL_CONFIG['model'],
    api_key=MODEL_CONFIG['api_key'],
    base_url=MODEL_CONFIG['base_url'],
)

print('模型实例创建成功')
print('model:', MODEL_CONFIG['model'])
print('base_url:', MODEL_CONFIG['base_url'])

模型实例创建成功
model: qwq
base_url: http://192.168.102.19:8082/v1


## 2. 创建 internet_search 工具

复用上一课的工具定义。

In [3]:
from tavily import TavilyClient

tavily_api_key = os.getenv('TAVILY_API_KEY', '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')

def internet_search(query: str) -> str:
    """Search the internet for information about the given query.

    Use this tool when you need to find information on the internet.
    Returns a formatted string with search results.

    Args:
        query: The search query string.

    Returns:
        Formatted search results as a string.
    """
    client = TavilyClient(api_key=tavily_api_key)
    results = client.search(query=query, max_results=5)

    formatted = []
    for i, result in enumerate(results.get('results', []), 1):
        title = result.get('title', 'N/A')
        url = result.get('url', 'N/A')
        content = result.get('content', 'N/A')
        formatted.append(f'[{i}] {title}\nURL: {url}\n{content}\n')

    return '\n'.join(formatted) if formatted else 'No results found.'

print('internet_search 工具创建成功')

internet_search 工具创建成功


## 3. 使用 create_deep_agent 创建 Agent

`create_deep_agent` 的核心参数：

| 参数 | 类型 | 作用 |
|---|---|---|
| model | ChatModel | 大模型实例，负责推理和决策 |
| tools | list[Tool] | 工具列表，供 agent 调用 |
| system_prompt | str | 系统提示，指导 agent 行为 |

如果用 Java 后端类比：

```text
model 像 Service 的核心处理器。
tools 像注入的依赖服务。
system_prompt 像配置类，定义处理规则。
```

In [4]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt='你是一个研究助手，擅长收集信息并撰写报告。',
)

print('Deep Agent 创建成功')
print('agent type:', type(agent).__name__)

Deep Agent 创建成功
agent type: CompiledStateGraph


## 4. System Prompt 设计

System Prompt 是 agent 的大脑设定。

好的 system prompt 应该：

1. **明确角色**：告诉 agent 它是什么
2. **说明能力**：告诉 agent 它能做什么
3. **定义边界**：告诉 agent 它不能做什么

示例：

```text
你是一个研究助手，擅长收集信息并撰写报告。
你可以使用 internet_search 工具搜索互联网。
请基于搜索结果撰写结构化报告。
```

不推荐：

```text
你是一个友好的助手。
```

In [5]:
prompt_examples = {
    '好的示例': (
        '你是一个研究助手，擅长收集信息并撰写报告。\n'
        '你可以使用 internet_search 工具搜索互联网。\n'
        '请基于搜索结果撰写结构化报告。'
    ),
    '不推荐': '你是一个友好的助手。',
}

for label, prompt in prompt_examples.items():
    print(f'=== {label} ===')
    print(prompt)
    print()

=== 好的示例 ===
你是一个研究助手，擅长收集信息并撰写报告。
你可以使用 internet_search 工具搜索互联网。
请基于搜索结果撰写结构化报告。

=== 不推荐 ===
你是一个友好的助手。



## 5. 本课小结

本课完成了：

1. 创建了 LangChain 模型实例
2. 创建了 `internet_search` 工具
3. 使用 `create_deep_agent` 创建了 agent
4. 理解了 system prompt 的设计原则

产出了：

- 一个配置好 model 和 tools 的 Deep Agent
- 理解了 `create_deep_agent` 的参数

## 下一课预告

下一课要做的是：

```text
运行 agent 并观察执行过程
查看 agent 的规划、工具调用、子代理委派事件
```

## 6. 练习

请你思考后回答：

1. `create_deep_agent` 的三个核心参数是什么？
2. 为什么 system prompt 需要明确角色、能力和边界？
3. 如果让你设计一个写代码的 agent，你会给它注册哪些工具？